In [1]:
# preprocessing_pipeline_cleaned.py
# Optimized Data Cleaning Pipeline for Fake News Datasets
# Replicates the exact cleaning logic used to generate the submitted datasets.

import os
import re
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings

warnings.filterwarnings('ignore')

# =============================================================================
# CONFIGURATION
# =============================================================================
# Update these paths to where your RAW data is located
RAW_ISOT_TRUE = "/kaggle/input/fake-news-detection-datasets/News _dataset/True.csv"
RAW_ISOT_FAKE = "/kaggle/input/fake-news-detection-datasets/News _dataset/Fake.csv"
RAW_TEXTDB3   = "/kaggle/input/textdb3/fake_or_real_news.csv" # Check exact name
RAW_WELFAKE   = "/kaggle/input/fake-news-classification/WELFake_Dataset.csv"

OUTPUT_DIR = "cleaned_dataset/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Similarity threshold for deduplication (Matches the original script's default)
SIMILARITY_THRESHOLD = 0.98 

# =============================================================================
# CLEANING UTILITIES
# =============================================================================
def clean_text(text):
    """
    Exact cleaning function from the original system.
    - Lowercase
    - Removes URLs, Emails, HTML
    - Removes all characters except Letters and Spaces (Removes numbers & punctuation)
    """
    if pd.isna(text):
        return ""
        
    text = str(text).lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # THE KEY LINE: Removes special characters and digits, keeps only letters/spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def remove_near_duplicates(df, text_column='text', similarity_threshold=0.98, batch_size=1000):
    """
    Removes near-duplicates using TF-IDF and Cosine Similarity.
    Matches the logic that reduced your ISOT/WELFake file sizes.
    """
    print(f"   🔍 Removing near-duplicates (similarity >= {similarity_threshold})...")
    original_size = len(df)
    
    if original_size == 0:
        return df
    
    df = df.reset_index(drop=True)
    
    # Create TF-IDF vectors (Limited features for speed, just like original)
    vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
    
    try:
        tfidf_matrix = vectorizer.fit_transform(df[text_column].astype(str))
    except Exception as e:
        print(f"   ⚠️ Vectorization failed: {e}")
        return df
    
    duplicates_to_remove = set()
    n_samples = tfidf_matrix.shape[0]
    
    # Process in batches
    for i in tqdm(range(0, n_samples, batch_size), desc="   Checking batches"):
        batch_end = min(i + batch_size, n_samples)
        similarities = cosine_similarity(tfidf_matrix[i:batch_end], tfidf_matrix)
        
        for local_idx, row_similarities in enumerate(similarities):
            global_idx = i + local_idx
            if global_idx in duplicates_to_remove:
                continue
            
            # Find matches ahead of current index
            duplicate_indices = np.where(row_similarities >= similarity_threshold)[0]
            duplicate_indices = duplicate_indices[duplicate_indices > global_idx]
            duplicates_to_remove.update(duplicate_indices.tolist())
    
    # Keep non-duplicates
    indices_to_keep = [i for i in range(n_samples) if i not in duplicates_to_remove]
    df_cleaned = df.iloc[indices_to_keep].reset_index(drop=True)
    
    removed = original_size - len(df_cleaned)
    print(f"   ✓ Removed {removed} duplicates ({removed/original_size*100:.2f}%)")
    
    return df_cleaned

# =============================================================================
# DATASET PROCESSORS
# =============================================================================
def process_isot():
    print("\n📚 Processing ISOT Dataset...")
    try:
        df_true = pd.read_csv(RAW_ISOT_TRUE)
        df_fake = pd.read_csv(RAW_ISOT_FAKE)
        
        df_true['label'] = 0
        df_fake['label'] = 1
        
        df = pd.concat([df_true, df_fake], ignore_index=True)
        
        # 1. Clean
        df['text'] = df['text'].apply(clean_text)
        df['title'] = df['title'].apply(clean_text)
        
        # 2. Deduplicate
        df = remove_near_duplicates(df, similarity_threshold=SIMILARITY_THRESHOLD)
        
        # 3. Add Metadata placeholders (Matches your file)
        # Note: We assign the dict as a string or object, pandas saves it as string representation
        df['fetch_metadata'] = [{'fetched': False, 'url': None} for _ in range(len(df))]
        df['image_features'] = None
        
        # Ensure column order
        cols = ['title', 'text', 'subject', 'date', 'label', 'fetch_metadata', 'image_features']
        for c in cols:
            if c not in df.columns: df[c] = None
        df = df[cols]
        
        save_path = os.path.join(OUTPUT_DIR, "cleaned_ISOT_Dataset.csv")
        df.to_csv(save_path, index=False)
        print(f"✅ Saved ISOT to {save_path}")
        
    except Exception as e:
        print(f"❌ Error processing ISOT: {e}")

def process_textdb3():
    print("\n📚 Processing TextDB3 Dataset...")
    try:
        df = pd.read_csv(RAW_TEXTDB3)
        
        # Handle column names
        rename_map = {col: col.lower() for col in df.columns}
        if 'headline' in rename_map.values():
            key = [k for k, v in rename_map.items() if v == 'headline'][0]
            df = df.rename(columns={key: 'title'})
            
        # Handle labels
        if df['label'].dtype == object:
            df['label'] = df['label'].map({'REAL': 0, 'FAKE': 1, 'real': 0, 'fake': 1})
        
        # 1. Clean
        df['text'] = df['text'].apply(clean_text)
        if 'title' in df.columns:
            df['title'] = df['title'].apply(clean_text)
            
        # 2. Deduplicate
        df = remove_near_duplicates(df, similarity_threshold=SIMILARITY_THRESHOLD)
        
        # 3. Add placeholders
        df['fetch_metadata'] = [{'fetched': False, 'url': None} for _ in range(len(df))]
        df['image_features'] = "[]" # Matches TextDB3 snippet format
        
        cols = ['title', 'text', 'label', 'fetch_metadata', 'image_features']
        for c in cols:
            if c not in df.columns: df[c] = None
            
        save_path = os.path.join(OUTPUT_DIR, "cleaned_TextDB3_Dataset.csv")
        df.to_csv(save_path, index=False)
        print(f"✅ Saved TextDB3 to {save_path}")
        
    except Exception as e:
        print(f"❌ Error processing TextDB3: {e}")

def process_welfake():
    print("\n📚 Processing WELFake Dataset...")
    try:
        df = pd.read_csv(RAW_WELFAKE)
        
        # Enforce unified labeling: 0 = Real, 1 = Fake
        df['label'] = df['label'].map({1: 0, 0: 1})
        df['label'] = df['label'].astype(int)

        # 1. Clean
        df['text'] = df['text'].apply(clean_text)
        df['title'] = df['title'].apply(clean_text)
        
        # 2. Deduplicate (Critical for WELFake as it has many dupes)
        df = remove_near_duplicates(df, similarity_threshold=SIMILARITY_THRESHOLD)
        
        # 3. Add placeholders
        df['fetch_metadata'] = [{'fetched': False, 'url': None} for _ in range(len(df))]
        df['image_features'] = "[]"
        
        save_path = os.path.join(OUTPUT_DIR, "cleaned_WELFake_Dataset.csv")
        df.to_csv(save_path, index=False)
        print(f"✅ Saved WELFake to {save_path}")
        
    except Exception as e:
        print(f"❌ Error processing WELFake: {e}")

# =============================================================================
# MAIN EXECUTION
# =============================================================================
if __name__ == "__main__":
    print("🚀 Starting Data Preprocessing Pipeline...")
    process_isot()
    process_textdb3()
    process_welfake()
    print("\n✨ All datasets processed and saved to /cleaned_dataset/")

🚀 Starting Data Preprocessing Pipeline...

📚 Processing ISOT Dataset...
   🔍 Removing near-duplicates (similarity >= 0.98)...


   Checking batches:   0%|          | 0/45 [00:00<?, ?it/s]

   ✓ Removed 5943 duplicates (13.24%)
✅ Saved ISOT to cleaned_dataset/cleaned_ISOT_Dataset.csv

📚 Processing TextDB3 Dataset...
   🔍 Removing near-duplicates (similarity >= 0.98)...


   Checking batches:   0%|          | 0/7 [00:00<?, ?it/s]

   ✓ Removed 311 duplicates (4.91%)
✅ Saved TextDB3 to cleaned_dataset/cleaned_TextDB3_Dataset.csv

📚 Processing WELFake Dataset...
   🔍 Removing near-duplicates (similarity >= 0.98)...


   Checking batches:   0%|          | 0/73 [00:00<?, ?it/s]

   ✓ Removed 9638 duplicates (13.36%)
✅ Saved WELFake to cleaned_dataset/cleaned_WELFake_Dataset.csv

✨ All datasets processed and saved to /cleaned_dataset/
